# Track 2: Cross-Venue Funding Rate & Basis Dispersion

Empirical cross-venue delta-neutral yield scanner across Binance, Bybit, and OKX.
Captures funding payouts every 8 hours with zero directional price risk.

In [ ]:
import asyncio
import ccxt.async_support as ccxt
import polars as pl

symbols = [
    "ADA/USDT:USDT", "LINK/USDT:USDT", "DOGE/USDT:USDT", "SUI/USDT:USDT",
    "ETH/USDT:USDT", "AVAX/USDT:USDT", "BTC/USDT:USDT", "NEAR/USDT:USDT",
    "APT/USDT:USDT", "XRP/USDT:USDT", "BNB/USDT:USDT", "SOL/USDT:USDT"
]

async def scan_live_dispersion():
    binance = ccxt.binance({"enableRateLimit": True})
    bybit = ccxt.bybit({"enableRateLimit": True})
    
    data = []
    try:
        for sym in symbols:
            b_r = (await binance.fetch_funding_rate(sym)).get("fundingRate", 0.0)
            by_r = (await bybit.fetch_funding_rate(sym)).get("fundingRate", 0.0)
            spread = abs(b_r - by_r)
            ann_apr = spread * 3.0 * 365.0 * 100.0
            setup = "Short Binance / Long Bybit" if b_r > by_r else "Long Binance / Short Bybit"
            data.append({
                "asset": sym,
                "binance_8h_pct": b_r * 100.0,
                "bybit_8h_pct": by_r * 100.0,
                "spread_8h_pct": spread * 100.0,
                "annualized_apr_pct": ann_apr,
                "strategy_setup": setup
            })
    finally:
        await binance.close()
        await bybit.close()
        
    df = pl.DataFrame(data).sort("annualized_apr_pct", descending=True)
    return df

# Run scanner
import nest_asyncio
nest_asyncio.apply()
df = asyncio.run(scan_live_dispersion())
print(df)